# Polarized observations

This tutorial covers working with polarized instrument and maps, and recovering polarized maps from observations.

We start with a normal instrument, and create two orthogonally polarized copies of each detector by setting ``polarized: True`` in the ``Array`` config:

In [ ]:
import maria
from maria.instrument import Band

f150 = Band(
    center=150e9, 
    width=30e9, 
    NET_RJ=60e-6, 
    knee=1e0, 
    gain_error=5e-2)

array = {"field_of_view": 0.2, 
         "shape": "circle", 
         "beam_spacing": 1.8,
         "primary_size": 10, 
         "polarized": True,
         "packing": "sunflower",
         "bands": [f150]}

instrument = maria.get_instrument(array=array)

print(instrument.arrays)

We can see the resulting polarization footprint in the instrument plot:

In [ ]:
print(instrument)
instrument.plot()

Let's observe the use the Einstein map, which has a faint polarization signature underneath the unpolarized signal of Einstein's face. Remember that all maps are five dimensional (stokes, frequency, time, y, x); this map has four channels in the stokes dimensions (the I, Q, U, and V [Stokes parameters](https://en.wikipedia.org/wiki/Stokes_parameters)). We can plot all the channels by giving ``plot`` a shaped set of stokes parameters.

In [ ]:
input_map = maria.map.get("maps/einstein.h5")
input_map.data *= 4
print(input_map)


input_map.plot(slices={"stokes": [["I", "Q"], 
                                  ["U", "V"]]})

In [ ]:
from maria import Planner

planner = Planner(target=input_map, site="mauna_kea", constraints={"el": (45, 90)})

plans = planner.generate_plans(
                               total_duration=1800,
                               max_chunk_duration=1800,
                               sample_rate=50, 
                               scan_pattern="raster",
                               scan_options={
                                   "n": [(15, 1), (1, 16)],
                                   "radius": 0.5 * input_map.width.deg,
                                   "speed": 0.5,
                               })

plans[0].plot()
print(plans)

In [ ]:
sim = maria.Simulation(
    instrument,
    plans=plans,
    site="mauna_kea",
    atmosphere="2d",
    map=input_map,
)

print(sim)

In [ ]:
tods = sim.run()

print(tods)
tods[0].plot()

In [ ]:
from maria.mappers import MaximumLikelihoodMapper

mapper = MaximumLikelihoodMapper(
    frame="ra/dec",
    resolution=2 * input_map.resolution.deg,
    tod_preprocessing={
        "remove_spline": {"knot_spacing": 300, "remove_el_gradient_order": 1},
    },
    tods=tods,
    bilinear=False,
)

mapper.map.plot(slices="all")

In [ ]:
mapper.fit(epochs=2, 
           steps_per_epoch=25, 
           plot=True, 
           plot_kwargs={
                        "slices": "all",
                        "center_zero": True,
                        "contrast": 1e-3})

Note that we can't see any of the circular polarization, since our instrument isn't sensitive to it.